In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 100),
            keep=("messages", 1) # keep the last message
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is engaging in a fictional, imaginative conversation about the moon, its fictional capital Lunapolis, its inhabitants (cheese miners), and potential labor disputes. The overall goal is to continue this creative narrative.\n\n## SUMMARY\nThe conversation has established the following:\n*   The capital of the moon is Lunapolis.\n*   The weather in Lunapolis is clear with a high of 120°C and a low of -100°C.\n*   There are 100,000 cheese miners in Lunapolis.\n*   The cheese miners' union is likely to strike due to unhappiness with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nContinue the imaginative conversation, potentially exploring the reasons for the cheese miners' unhappiness or the implications of a strike.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='888f7dd7-4ebf-4aea-a510-3e6e6f214ca9'),
              HumanMessage(content

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is engaging in a fictional, imaginative conversation about the moon, its fictional capital Lunapolis, its inhabitants (cheese miners), and potential labor disputes. The overall goal is to continue this creative narrative.

## SUMMARY
The conversation has established the following:
*   The capital of the moon is Lunapolis.
*   The weather in Lunapolis is clear with a high of 120°C and a low of -100°C.
*   There are 100,000 cheese miners in Lunapolis.
*   The cheese miners' union is likely to strike due to unhappiness with the new president.

## ARTIFACTS
None

## NEXT STEPS
Continue the imaginative conversation, potentially exploring the reasons for the cheese miners' unhappiness or the implications of a strike.


## Trim/delete messages

In [6]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [8]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [9]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='09a5d3c1-715a-4013-a845-1ae7ecbee95f'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='05efb0ff-1b5e-444c-8f7c-741788e27eea', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='7062c5ec-4ba5-40db-b74f-4b0cbdd93f63'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='dc7cb2e6-4d7a-4deb-bbe3-08bce158bc06', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='c72c4baf-be9e-42d6-9f33-b5919565c26e'),
              AIMessage(content="I can't tell you the temperature of your device. I am a large 

In [10]:
print(response["messages"][-1].content)

I can't tell you the temperature of your device. I am a large language model, able to communicate in response to a wide range of prompts and questions, but my knowledge about that is limited. Is there anything else I can do to help?
